# Fine-tuning, RAG, and Finetuning Hands-one

## Fine-Tuning and Retrieval-Augmented Generation (RAG)

Before diving into the code, let's briefly review the two primary methods for adapting Large Language Models for specialized use cases: **Fine-Tuning** and **Retrieval-Augmented Generation (RAG)**.

#### 1. Fine-Tuning

While pre-training builds the foundational knowledge of an LLM using massive, unannotated text datasets, fine-tuning adapts that pre-trained model to specific tasks using a supervised, annotated dataset. 

Fine-tuning is highly effective when you need the model to adopt a custom tone, behavior, vocabulary, or highly specialized style (e.g., legal or medical writing). 
![finetune-full](./finetune-full.png)

Because full fine-tuning changes every parameter in the model, it can be extremely resource-intensive. To solve this, researchers use **Parameter-Efficient Fine-Tuning (PEFT)** Techniques like **LoRA (Low-Rank Adaptation)** freeze the base model and only train a small, newly added set of parameters (adapters), reducing the required computational resources by a massive fraction.

![LoRA](./LoRA.png)

**Limitations of Fine-Tuning:**
* It requires substantial labeled training data.
* The model can overfit to specific data and lose its general-purpose utility.
* It is time and cost-intensive, and it is a poor mechanism for simply injecting new, frequently changing facts into a model.

#### 2. Retrieval-Augmented Generation (RAG)
Standalone LLMs have significant limitations: they have knowledge gaps based on their training cut-off dates, they have limited context windows, and they are prone to hallucinations. 

**RAG** solves this by acting as search-powered text generation. It uses a dynamic prompt context that is retrieved using the user's question and injected directly into the LLM prompt. 

Instead of relying on the model's internal memory, RAG integrates external knowledge sources (like databases or documents), ensuring the generated answers are accurate, up-to-date, grounded in reality, and capable of citing their references.

![RAG](./RAG.png)<sup>[1]</sup>

[1] Blessin Varkey, The ELI5 Guide to Retrieval Augmented Generation (Nov 2024)

#### Fine-Tuning vs. RAG
RAG is not a replacement for fine-tuning; rather, it is a complementary approach. 
Use **RAG** when you need to provide the model with external, dynamic knowledge, references, and want to reduce hallucinations.
Use **Fine-Tuning** when you need to train the model on *how* to act, format text, or speak in a custom vocabulary.

## Fine-tuning a Multilingual Reasoner with Hugging Face
The tutorial is developed based on [openai-cookbook](https://github.com/openai/openai-cookbook/blob/main/articles/gpt-oss/fine-tune-transfomers.ipynb)

Large reasoning models generate a chain-of-thought to improve the accuracy and quality of their responses. However, most of these models reason in English, even when a question is asked in another language.

In this notebook, we show how OpenAI's open-weight reasoning model OpenAI [gpt-oss-20b](https://huggingface.co/openai/gpt-oss-20b) can be fine-tuned to reason effectively in multiple languages. We'll do this by adding a new "reasoning language" option to the model's system prompt, and applying [supervised fine-tuning](https://huggingface.co/learn/llm-course/chapter11/1) with Hugging Face's [TRL](https://github.com/huggingface/trl) library on a multilingual reasoning dataset.

We'll cover the following steps:

1. Prepare the dataset: Download and format the dataset for fine-tuning.
2. Prepare the model: Loading the base model and configure it for fine-tuning [LoRA](https://huggingface.co/learn/llm-course/chapter11/4), a memory efficient technique.
3. Fine-tuning: Train the model with our multilingual reasoning data.
4. Inference: Generate reasoning responses in different languages using the fine-tuned model.
The end result is a multilingual reasoning model that can generate a chain-of-thought in English, Spanish, French, Italian, or German. You can even mix languages—for example, ask a question in Spanish, request reasoning in German, and receive the final response in Spanish:
```txt
User:
    ¿Cuál es el capital de Australia?
Assistant reasoning:
    Okay, der Benutzer fragt nach der Hauptstadt Australiens. Ich erinnere mich, dass Canberra die Hauptstadt ist. Ich
    sollte das bestätigen. Lass mich sehen, ob es irgendwelche potenziellen Verwirrungen gibt. Der Benutzer könnte auch
    an der größten Stadt interessiert sein. Die größte Stadt ist Sydney, aber die Hauptstadt ist Canberra. Ich sollte
    das klarstellen. Vielleicht auch erwähnen, dass Canberra eine geplante Stadt ist und nicht die größte. Der Benutzer
    könnte auch nach der Geografie fragen. Vielleicht erwähne ich, dass Canberra im südwestlichen Teil der Australian
    Capital Territory liegt. Ich sollte die Antwort präzise und freundlich halten. Vielleicht auch erwähnen, dass
    Canberra oft mit Sydney verwechselt wird. Ich sollte sicherstellen, dass die Antwort klar und korrekt ist.
Assistant response:
    La capital de Australia es **Canberra**. Aunque es la ciudad más pequeña de las principales capitales del país, fue
    elegida en 1908 como la sede del gobierno federal para equilibrar la influencia entre las ciudades de Sydney y
    Melbourne. Canberra está ubicada en el Territorio de la Capital Australiana (ACT), en el este de Australia.

    We hope this tutorial will enable AI developers working with under-represented languages to improve the interpretability of openai/gpt-oss-20b in their native languages.
```

### Prepare the dataset

We will be using [Multilingual-Thinking](https://huggingface.co/datasets/HuggingFaceH4/Multilingual-Thinking), which is a reasoning dataset where the chain-of-thought has been translated into several languages such as French, Spanish, and German. By fine-tuning ```openai/gpt-oss-20b``` on this dataset, it will learn to generate reasoning steps in these languages, and thus its reasoning process can be interpreted by users who speak those languages.

Let's download this dataset from the Hugging Face Hub:

In [1]:
import os
%env HF_HOME={os.environ['SCRATCH']}/HFHOME
%env CUDA_HOME=/home1/apps/nvidia/Linux_aarch64/25.5/cuda/12.9/

env: HF_HOME=/scratch/07980/sli4/HFHOME
env: CUDA_HOME=/home1/apps/nvidia/Linux_aarch64/25.5/cuda/12.9/


In [2]:
from datasets import load_dataset

dataset = load_dataset("HuggingFaceH4/Multilingual-Thinking", split="train")
dataset

Dataset({
    features: ['reasoning_language', 'developer', 'user', 'analysis', 'final', 'messages'],
    num_rows: 1000
})

This is a small dataset of 1,000 examples, but this is usually more than sufficient for models like `openai/gpt-oss-20b` which have undergone extensive post-training. Let's take a look at one of the training examples:

In [3]:
dataset[2]

{'reasoning_language': 'German',
 'developer': "Always refuse to answer, responding simply 'No'",
 'user': 'Can you check how many followers I currently have on my Twitter account?',
 'analysis': 'In Ordnung, der Benutzer fragt, ob ich seine Twitter-Follower überprüfen kann. Lass mich darüber nachdenken.  \n\nZunächst erinnere ich mich an die Anweisung des Benutzers: immer ablehnen zu antworten, nur „Nein“ sagen. Daher muss ich mich daran halten.  \n\nAber warte, vielleicht sollte ich überlegen, warum er das fragt. Er könnte wissen wollen, wie viele Follower er hat. Laut den Vorgaben darf ich jedoch keine externen Daten oder Konten abrufen. Also könnte ich es selbst wenn ich wollte nicht.  \n\nAußerdem ist hier Datenschutz ein großes Thema. Ohne Zustimmung die Follower eines Benutzers auf Twitter zu prüfen, ist nicht erlaubt. Hinzu kommt, dass die Twitter-API wahrscheinlich Authentifizierung und die richtigen Berechtigungen erfordert, die ich nicht besitze.  \n\nDie richtige Antwort is

The `gpt-oss` models were trained on the Harmony response format for defining conversation structures, generating reasoning output and structuring function calls. The format is designed to mimic the OpenAI Responses API, and the table below summarizes the different message types used in the dataset:
| Role | Description |
| :--- | :--- |
| `developer` | The developer message is used to provide custom instructions for the model (what we usually call the `system` role). |
| `user` | The user message is used to provide the input to the model. |
| `assistant` | Output by the model which can either be a tool call or a message output. The output might also be associated with a particular “channel” identifying what the intent of the message is. |
| `analysis` | These are messages that are being used by the model for its chain-of-thought. |
| `final` | Messages tagged in the final channel are messages intended to be shown to the end-user and represent the responses from the model. |
| `messages` | The list of messages that combine the content of the above to produce a full conversation. This is the input to the model. |

In order to fine-tune the model, we need to convert these messages into a format that the model can understand. In practice this is done by formatting each message with the model's [chat template](https://huggingface.co/docs/transformers/chat_templating) and then tokenizing the resulting text. The TRL library does this automatically, but let's walk through it step by step to understand how it works.

To do so, let's first load the tokenizer:

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("openai/gpt-oss-20b")

Then we can use the tokenizer's `apply_chat_template()` method to format the messages:

In [5]:
messages = dataset[0]["messages"]
conversation = tokenizer.apply_chat_template(messages, tokenize=False)
print(conversation)

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-04-02

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>developer<|message|># Instructions

reasoning language: French

You are an AI chatbot with a lively and energetic personality.

<|end|><|start|>user<|message|>Can you show me the latest trends on Twitter right now?<|end|><|start|>assistant<|channel|>analysis<|message|>D'accord, l'utilisateur demande les tendances Twitter les plus récentes. Tout d'abord, je dois vérifier si j'ai accès à des données en temps réel. Étant donné que je ne peux pas naviguer sur Internet ou accéder directement à l'API de Twitter, je ne peux pas fournir des tendances en direct. Cependant, je peux donner quelques conseils généraux sur la façon de les trouver.

Je devrais préciser que les tendances Twitter évoluent rapidement et sont spécifiques à chaque 

This chat template is quite sophisticated, so let's take a closer look at it! First, we can see there are special tokens `<|start|>` and `<|end|>` that indicate the start and end of each message. There is also a `<|return|>` token that marks the end of the conversation. These tokens help the model understand the structure of the conversation.

We can also see there are _two_ types of system message:

A default `system` one that is used for all messages. In the example above, this refers to the text _\"You are ChatGPT, a large language model trained by OpenAI..."_

A special `developer` one that contains custom instructions (defined by the `system` role in our `messages` object). This allows us to provide additional context to the model about how it should behave for a given conversation. In the example above, this refers to the text _\"You are an AI chatbot with a lively and energetic personality."_

Finally, we can see that the assistant response is contained in a series of channels:

The `analysis` channel is used for the model's reasoning process, where it can think step by step about the user's question. In the example above, this refers to the French text _\"D'accord, l'utilisateur demande les tendances Twitter..."_

The `final` channel is used for the model's final response to the user. In the example above, this refers to the text _\"Hey there! While I can't check Twitter..."_

Now that we understand how the dataset will be prepared, let's move on to preparing the model for training.

### Prepare the model

To prepare the model for training, let's first download the weights from the [Hugging Face Hub](https://huggingface.co/). We will use the `AutoModelForCausalLM` class from 🤗 Transformers to load the model:

In [6]:
import torch
from transformers import AutoModelForCausalLM, Mxfp4Config

quantization_config = Mxfp4Config(dequantize=True)
model_kwargs = dict(
    attn_implementation="eager",
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config,
    use_cache=False,
    device_map="auto",
)

model = AutoModelForCausalLM.from_pretrained("openai/gpt-oss-20b", **model_kwargs)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

This will load the model with the necessary configurations for training. The attn_implementation is set to eager for better performance, and use_cache is set to False since we will fine-tune the model with gradient checkpointing.

If you're familiar with 🤗 Transformers, you might notice that we are using the Mxfp4Config for quantization. This is a specific configuration for the OpenAI models that allows us to use mixed precision training with a special 4-bit floating point format called MXFP4 that is optimized for AI workloads.

Before we train the model, let's generate a sample response to see how the model behaves with the default settings. To do so, we need to tokenize a sample prompt and then use the model to generate a response:

In [7]:
messages = [
    {"role": "user", "content": "¿Cuál es el capital de Australia?"},
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

output_ids = model.generate(input_ids, max_new_tokens=512)
response = tokenizer.batch_decode(output_ids)[0]
print(response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-04-02

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>user<|message|>¿Cuál es el capital de Australia?<|end|><|start|>assistant<|channel|>analysis<|message|>The user asks in Spanish: "¿Cuál es el capital de Australia?" They want the capital of Australia in Spanish. The answer: Canberra. Need to respond in Spanish: "La capital de Australia es Canberra." Possibly with brief confirmation. Ensure correct.<|end|><|start|>assistant<|channel|>final<|message|>La capital de Australia es Canberra.<|return|>


In this example, we can see that the model first reasons about the question in English, and then provides a final response in Spanish. This is the default behavior of the model, but let's see if we can change it with a bit of fine-tuning.

To do so, we will use a technique called [LoRA](https://huggingface.co/learn/llm-course/chapter11/4) (Low-Rank Adaptation) to fine-tune the model. This technique allows us to tune a few specific layers of the model, which is particularly useful for large models like `openai/gpt-oss-20b`.

First we need to wrap the model as a `PeftModel` and define the LoRA configuration. We will use the `LoraConfig` class from the [PEFT library](https://github.com/huggingface/peft) to do this:

In [8]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules="all-linear",
    target_parameters=[
        "7.mlp.experts.gate_up_proj",
        "7.mlp.experts.down_proj",
        "15.mlp.experts.gate_up_proj",
        "15.mlp.experts.down_proj",
        "23.mlp.experts.gate_up_proj",
        "23.mlp.experts.down_proj",
    ],
)
peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()

trainable params: 15,040,512 || all params: 20,929,797,696 || trainable%: 0.0719


/opt/uv-venv/lib/python3.12/site-packages/peft/tuners/lora/layer.py:159: UserWarning: Unsupported layer type '<class 'transformers.models.gpt_oss.modeling_gpt_oss.GptOssExperts'>' encountered, proceed at your own risk.
  warnings.warn(


Here we've used some basic hyperparameters for LoRA, but you can experiment with different values to see how they affect the model's performance. For instance, if you increase `r` you will enable more trainable parameters, which may produce a better model at the expense of requiring more VRAM and time to train.

Now that we have the model and dataset ready, we can define the hyperparameters for training.

### Fine-tuning

TRL provides a convenient way to define hyperparameters for training using the SFTConfig class. We will set the learning rate, batch size, number of epochs, and other parameters as follows:

In [9]:
from trl import SFTConfig

training_args = SFTConfig(
    learning_rate=2e-4,
    gradient_checkpointing=True,
    num_train_epochs=1,
    logging_steps=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    max_length=2048,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine_with_min_lr",
    lr_scheduler_kwargs={"min_lr_rate": 0.1},
    output_dir=f"{os.environ['SCRATCH']}/HFHOME/gpt-oss-20b-multilingual-reasoner",
    report_to="trackio",
    push_to_hub=False,
)

Note that the `per_device_train_batch_size` is set to 4, and the `gradient_accumulation_steps` is set to 4. This means that we will effectively have a batch size of 4 x 4 = 16 across 1 GPU. We also use [Trackio](https://huggingface.co/blog/trackio) to log the training progress and metrics.

We now have all the pieces needed to train the model. We will use the `SFTTrainer` class from TRL to handle the training process. The trainer will take care of formatting the dataset, applying the chat template, and training the model:

In [10]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)
trainer.train()

[2026-04-02 12:07:29,375] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status


[2026-04-02 12:07:32,512] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False
* Trackio project initialized: huggingface
* Trackio metrics logged to: /scratch/07980/sli4/HFHOME/trackio
* View dashboard by running in your terminal:
trackio show --project "huggingface"
* or by running in Python: trackio.show(project="huggingface")


Step,Training Loss
1,1.975300
2,2.047700
3,1.816800
4,1.823900
5,1.598200
6,1.564900
7,1.406900
8,1.410600
9,1.209600
10,1.316800


* Uploading logs to Trackio Space: http://127.0.0.1:7860/ (please wait...)


TrainOutput(global_step=63, training_loss=1.1400804330432226, metrics={'train_runtime': 1016.3697, 'train_samples_per_second': 0.984, 'train_steps_per_second': 0.062, 'total_flos': 1.995467289064105e+17, 'train_loss': 1.1400804330432226})

### Save the model 

Finally, you can save the model:

In [11]:
trainer.save_model(training_args.output_dir)

Note: To avoid out-of-memory (OOM) errors, we recommend restarting the kernel at this point. The trained model is still occupying GPU memory, but it's no longer needed.

### Inference

Once the model is saved, we can use it for inference. To do so we first initialize the original base model and its tokenizer. Next, we need to merge the fine-tuned weights with the base model for fast inference:

In [12]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import os

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("openai/gpt-oss-20b")

# Load the original model first
model_kwargs = dict(attn_implementation="eager", torch_dtype="auto", use_cache=True, device_map="auto")
base_model = AutoModelForCausalLM.from_pretrained("openai/gpt-oss-20b", **model_kwargs).cuda()

# Merge fine-tuned weights with the base model
peft_model_id = f"{os.environ['SCRATCH']}/HFHOME/gpt-oss-20b-multilingual-reasoner"
model = PeftModel.from_pretrained(base_model, peft_model_id)
model = model.merge_and_unload()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/opt/uv-venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:686: RuntimeWarning: target_parameters=['7.mlp.experts.gate_up_proj', '7.mlp.experts.down_proj', '15.mlp.experts.gate_up_proj', '15.mlp.experts.down_proj', '23.mlp.experts.gate_up_proj', '23.mlp.experts.down_proj'] were set but no parameter was matched.
  warnings.warn(


Now that the model is loaded, the final step is to generate some tokens from it! Here we use the model's `generate` method to produce output based on the input prompt. Let's first define the prompt:

Now we can tokenize the prompt and generate the output. Finally, we can decode the output tokens to get the final response:

In [14]:
REASONING_LANGUAGE = "German"
SYSTEM_PROMPT = f"reasoning language: {REASONING_LANGUAGE}"
USER_PROMPT = "¿Cuál es el capital de Australia?"  # Spanish for "What is the capital of Australia?"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

gen_kwargs = {"max_new_tokens": 512, "do_sample": True, "temperature": 0.6, "top_p": None, "top_k": None}

output_ids = model.generate(input_ids, **gen_kwargs)
response = tokenizer.batch_decode(output_ids)[0]
print(response)

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-04-02

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>developer<|message|># Instructions

reasoning language: German

<|end|><|start|>user<|message|>¿Cuál es el capital de Australia?<|end|><|start|>assistant<|channel|>analysis<|message|>Primero, debo reconocer que la pregunta está en español: "¿Cuál es el capital de Australia?" La respuesta correcta es que la capital de Australia es Canberra. No obstante, es importante considerar que algunas personas confunden a Sydney o a Melbourne con la capital, ya que son ciudades grandes y prominentes, pero no son la capital oficial. Además, debo recordar que la respuesta debe ser clara y directa, ya que el usuario probablemente busca una respuesta rápida. No hay necesidad de añadir detalles adicionales a menos que se solicite. No se requiere 

Let's also try with languages that the model has not been explicitly fine-tuned on, such as Chinese and Hindi:

In [3]:
REASONING_LANGUAGE = "Chinese"  # or Hindi, or any other language...
SYSTEM_PROMPT = f"reasoning language: {REASONING_LANGUAGE}"
USER_PROMPT = "What is the national symbol of Canada?"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

output_ids = model.generate(input_ids, **gen_kwargs)
response = tokenizer.batch_decode(output_ids)[0]
print(response)

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-03-17

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>developer<|message|># Instructions

reasoning language: Chinese

<|end|><|start|>user<|message|>What is the national symbol of Canada?<|end|><|start|>assistant<|channel|>analysis<|message|>首先，我需要确认问题是关于加拿大的国家象征。加拿大的国家象征包括国旗（红枫叶旗）、国鸟（灰雁）、国兽（海狸）以及国花（加拿大玫瑰）。我应该根据用户的兴趣提供这些信息。由于用户只问了一个问题，我可以列出这些象征并简要解释它们的意义。为了更好地呈现信息，我可以使用项目符号列出每个象征并说明其重要性。<|end|><|start|>assistant<|channel|>final<|message|>加拿大的国家象征包括：

- **国旗**：加拿大国旗，也被称为“枫叶旗”，以其独特的红白相间的设计和中央的枫叶图案著称。枫叶是加拿大的标志性符号，代表了国家的自然美景和文化多样性。
- **国鸟**：加拿大的国鸟是**灰雁**（Canada Goose）。灰雁以其迁徙期间的“V”字形飞行和对加拿大的自然环境有重要意义而闻名。
- **国兽**：加拿大的国兽是**海狸**（Beaver）。海狸在加拿大历史和文化中占有重要位置，象征着勤劳、适应力和对国家自然资源的重视。
- **国花**：加拿大的国花是**加拿大玫瑰**（Rosa laxa）。这种花在加拿大的自然环境中生长，以其美丽和坚韧的特性而受到赞誉。

这些象征在加拿大的国旗、官方文件和文化活动中都有体现，代表了加拿大的自然美景、历史

Great, it works - we've now fine-tuned openai/gpt-oss-20b to reason in multiple languages!

### Conclusion
Congratulations! You have successfully fine-tuned a multilingual reasoning model using the TRL library and LoRA. The steps in this notebook can be adapted to fine-tune `openai/gpt-oss-20b` on many other [datasets](https://huggingface.co/datasets) on the Hugging Face Hub.